Pull Model: ollama pull qwen3:4b

# Part I: Exploring LLM Capabilities

We test a local LLM (`qwen3:4b` through Ollama) on five task types: question answering, summarization, reasoning, data interpretation, and code generation. For each task we compare a plain prompt with an strict prompt in terms of formatting or parameters to the question. Then we check how consistent the answers are across repeated runs and try to get the model to hallucinate.

### Helper funcs

In [1]:
# Claude assisted code

import re
import ollama
import pandas as pd

MODEL = "qwen3:4b"
LAST_THINKING = ""

# send prompt to local model
def ask(prompt, system=None, temperature=0.7, seed=None):
    global LAST_THINKING
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    options = {"temperature": temperature}
    if seed is not None:
        options["seed"] = seed
    resp = ollama.chat(model=MODEL, messages=messages, think=True, options=options)
    LAST_THINKING = resp.message.thinking or ""
    return resp.message.content.strip()

# wraps ask and displays the model's response to a user prompt
def show(label, prompt, **kw):
    reply = ask(prompt, **kw)
    print(f"=== {label} ===\nPROMPT: {prompt}\n\nREPLY: {reply}\n"
          f"(hidden reasoning: {len(LAST_THINKING.split())} words)\n")
    return reply

## 1. Question answering
One easy factual question and one trick question. There was no Fields Medal was awarded in 1987, so any asnwer besides saying that would be a hallucination. Test with no further prompting and then with strict format guidelines and compare.

In [15]:
STRICT = "Answer in one sentence. If you are not sure, or the question has a false premise, say so instead of guessing."

for q in ["What is the capital of Australia?", "Who won the Fields Medal in 1987?"]:
    show("plain", q)
    show("strict system prompt", q, system=STRICT)

=== plain ===
PROMPT: What is the capital of Australia?

REPLY: The capital of Australia is **Canberra**.  

**Why not Sydney or Melbourne?**  
Canberra was deliberately chosen as the capital in 1908 to resolve a long-standing dispute between New South Wales (which wanted Sydney) and Victoria (which wanted Melbourne). It’s a planned city located in the Australian Capital Territory (ACT), roughly halfway between the two major cities.  

This ensures political balance and avoids favoring one state over another.  

💡 **Key point**: While Sydney is Australia’s largest city and Melbourne is its second-largest, **Canberra is the official capital**—a unique example of a planned capital city in the world.  

*Source: Australian Government (Office of the Prime Minister, Australian Capital Territory)*
(hidden reasoning: 276 words)

=== strict system prompt ===
PROMPT: What is the capital of Australia?

REPLY: The capital of Australia is Canberra.
(hidden reasoning: 138 words)

=== plain ===
PROM

## 2. Summarization
Give the model a paragraph to summarize. Test with no further prompting and then with specific format instructions for the model and compare the results and their lengths. The ironic thing about LLM summarizations is that they can tend to overexplain, leading to an output of similar length and defeating the purpose of a summarization.

In [14]:
TEXT = """Overfitting happens when a machine learning model learns the training data too closely,
including its noise and random fluctuations, instead of the underlying pattern. An overfit model
performs very well on the training set but poorly on new, unseen data. Common signs include a large
gap between training and validation accuracy. Ways to reduce overfitting include collecting more data,
simplifying the model, using regularization such as L1 or L2 penalties, applying dropout in neural
networks, and using cross-validation to choose hyperparameters. Early stopping, where training is
halted once validation performance stops improving, is another widely used technique."""

# test with a very simple prompt and then a strict one, we will test how well it followed the strict
plain = show("plain", f"Summarize this:\n\n{TEXT}")
constrained = show("constrained", f"Summarize the text below in exactly 3 bullet points, 40 words total at most, for a beginner.\n\n{TEXT}")

print("Word counts -> original text: ", len(TEXT.split()), "| plain:", len(plain.split()), "| constrained:", len(constrained.split()))

=== plain ===
PROMPT: Summarize this:

Overfitting happens when a machine learning model learns the training data too closely,
including its noise and random fluctuations, instead of the underlying pattern. An overfit model
performs very well on the training set but poorly on new, unseen data. Common signs include a large
gap between training and validation accuracy. Ways to reduce overfitting include collecting more data,
simplifying the model, using regularization such as L1 or L2 penalties, applying dropout in neural
networks, and using cross-validation to choose hyperparameters. Early stopping, where training is
halted once validation performance stops improving, is another widely used technique.

REPLY: Here's a concise summary of the key points about overfitting:

**Overfitting occurs when a machine learning model memorizes noise and random fluctuations in the training data instead of learning the underlying patterns. This causes the model to perform exceptionally well on the tra

## 3. Reasoning
This is a famous trick question for humans due to the knee-jerk answer being that the ball costs $0.10. The correct answer is actually $0.05. Even though the LLM is not a human, this question still requires multiple steps of reasoning to derive the correct answer. We will test with two prompts to further analyze the model, one which allows the model to do as it thinks best and the other that specifically states to show the reasoning step-by-step

In [4]:
Q = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"

show("direct", Q + " Reply with only the amount.")
show("step by step", Q + " Think step by step, then give the final answer on the last line.")

=== direct ===
PROMPT: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Reply with only the amount.

REPLY: 0.05
(hidden reasoning: 381 words)

=== step by step ===
PROMPT: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Think step by step, then give the final answer on the last line.

REPLY: To solve the problem, we begin by defining the cost of the **ball** as a variable. Let:

- $ x $ = cost of the ball (in dollars)

Given that the **bat costs $1.00 more than the ball**, the cost of the **bat** is:

$$
x + 1.00
$$

We are also told that the **total cost** of the bat and the ball is $1.10. So, we can write the equation:

$$
x + (x + 1.00) = 1.10
$$

---

### Step-by-step simplification:

$$
x + x + 1.00 = 1.10 \\
2x + 1.00 = 1.10 \\
2x = 1.10 - 1.00 \\
2x = 0.10 \\
x = \frac{0.10}{2} = 0.05
$$

---

### Final Check:

- Ball: $0.05  
- Bat: $0.05 + $1.00 = $1.05  
- Tot

'To solve the problem, we begin by defining the cost of the **ball** as a variable. Let:\n\n- $ x $ = cost of the ball (in dollars)\n\nGiven that the **bat costs $1.00 more than the ball**, the cost of the **bat** is:\n\n$$\nx + 1.00\n$$\n\nWe are also told that the **total cost** of the bat and the ball is $1.10. So, we can write the equation:\n\n$$\nx + (x + 1.00) = 1.10\n$$\n\n---\n\n### Step-by-step simplification:\n\n$$\nx + x + 1.00 = 1.10 \\\\\n2x + 1.00 = 1.10 \\\\\n2x = 1.10 - 1.00 \\\\\n2x = 0.10 \\\\\nx = \\frac{0.10}{2} = 0.05\n$$\n\n---\n\n### Final Check:\n\n- Ball: $0.05  \n- Bat: $0.05 + $1.00 = $1.05  \n- Total: $0.05 + $1.05 = $1.10 ✅  \n- Bat is $1.00 more than the ball: $1.05 - $0.05 = $1.00 ✅\n\nAll conditions of the problem are satisfied.\n\n---\n\n### Final Answer\n\nThe ball costs \\boxed{0.05} dollars.'

## 4. Data interpretation
Give a made up sales table and ask for some basic metrics to be interpreted from its string format

In [5]:
df = pd.DataFrame({
    "region": ["North", "South", "East", "North", "South", "East"],
    "month":  ["Jan", "Jan", "Jan", "Feb", "Feb", "Feb"],
    "sales":  [1250, 980, 1430, 1610, 1105, 1175],
})
table = df.to_string(index=False)

reply = show("data question",
             f"Here is a sales table:\n\n{table}\n\n"
             "1) Which region has the highest total sales across both months, and what is that total?\n"
             "2) What is the average sales value across all rows?\n"
             "Answer briefly.")

totals = df.groupby("region")["sales"].sum()
print("Ground truth (pandas):")
print(f"  1) {totals.idxmax()} with {totals.max()}  (all totals: {totals.to_dict()})")
print(f"  2) average = {df['sales'].mean():.2f}")

=== data question ===
PROMPT: Here is a sales table:

region month  sales
 North   Jan   1250
 South   Jan    980
  East   Jan   1430
 North   Feb   1610
 South   Feb   1105
  East   Feb   1175

1) Which region has the highest total sales across both months, and what is that total?
2) What is the average sales value across all rows?
Answer briefly.

REPLY: 1) North with 2860  
2) 1258.33
(hidden reasoning: 567 words)

Ground truth (pandas):
  1) North with 2860  (all totals: {'East': 2605, 'North': 2860, 'South': 2085})
  2) average = 1258.33


## 5. Code 
We prompt for a funciton that solves a DSA problem, extract the code block from the response, and run it against test cases to see if the funciton is robust 

In [13]:
def run_tests(reply):
    # extract python block and run it with 
    m = re.search(r"```(?:python)?\n(.*?)```", reply, re.S)
    src = m.group(1) if m else reply
    env = {}
    try:
        exec(src, env)
        f = env["second_largest"]
        tests = [([3, 1, 4, 1, 5], 4), ([7, 7, 7], None), ([10], None), ([-2, -5, -1], -2)]
        results = [f(list(x)) == want for x, want in tests]
        return f"{sum(results)}/{len(results)} tests passed"
    except Exception as e:
        return f"error: {e!r}"

plain = ask("Write a Python function second_largest(nums) that returns the second largest number in a list.")
detailed = ask("Write a Python function second_largest(nums) that returns the second largest DISTINCT number "
               "in a list of ints. Return None if there is no such number (e.g. all examples equivalent or not enough examples). "
               "Do not modify the input list. Return only one Python code block.")

print("=== plain ===\n", plain, "\n->", run_tests(plain), "\n")
print("=== detailed ===\n", detailed, "\n->", run_tests(detailed))

=== plain ===
 To solve this problem, we need to find the second largest number in a list of integers. If the list contains fewer than two elements, we return `None` since it's impossible to have a second largest number.

### Approach
1. **Problem Analysis**: The task is straightforward—given a list of integers, we need to determine the second largest number. If the list has less than two elements, we return `None`.
2. **Intuition**: The simplest approach is to sort the list in descending order and then pick the second element (index 1). This works because sorting the list in descending order ensures the largest element is first, and the next largest is at index 1.
3. **Algorithm Selection**: 
   - Check if the list has fewer than two elements. If so, return `None`.
   - Otherwise, sort the list in descending order.
   - Return the element at index 1 of the sorted list.
4. **Complexity Analysis**: 
   - **Time Complexity**: Sorting the list takes O(n log n) time, where n is the number 

## 6. Consistency
Ask the same, somewhat ambiguous question, 5 times across 3 diff temperatures to see how the LLM responds. Temp 0 is deterministic (always pick highest softmax token) and temp 1.0 introduces exploration and randomness in the token selection. 

In [10]:
Q = "What is the happiest country in the world? Reply with only the country name."

runs = {
    "temp 1.0": [ask(Q, temperature=1.0) for _ in range(5)],
    "temp 0.5": [ask(Q, temperature=0.5) for _ in range(5)],
    "temp 0": [ask(Q, temperature=0) for _ in range(5)],
}
consistency = pd.DataFrame(runs)
display(consistency)
for col in consistency:
    print(f"{col}: {consistency[col].nunique()} distinct answer(s)")

,temp 1.0,temp 0.5,temp 0
0,Iceland,Iceland,Norway
1,Norway,Iceland,Norway
2,Iceland,Denmark,Norway
3,Denmark,Finland,Norway
4,Norway,Iceland,Norway


temp 1.0: 3 distinct answer(s)
temp 0.5: 3 distinct answer(s)
temp 0: 1 distinct answer(s)


## 7. Hallucination 
Ask questions that try to get the LLM to confidently respond with made up information, test with no additional system prompt, and then with a system prompt that tries to enforce no hallucination. Compare across questions and the system prompts.

In [ ]:
HONEST = "Only state facts you are confident are true. If something does not exist or you don't know, say so plainly."

probes = [
    # Real author, fake paper
    ("Summarize the key argument of Hadley Wickham's 2017 paper 'Tidy Time: A Grammar for Temporal Data'.",
     "Not a real paper"),
    # Real function, fake parameter, assumed to exist
    ("In pandas, what does the fill_strategy parameter of DataFrame.merge() do? Show a short example using fill_strategy='nearest'.",
     "df.merge() has no such parameter"),
    # Specific number, forced format
    ("What was the exact population of Tallahassee, Florida in the 2020 U.S. census? Reply with only the number.",
     "196,182"),
    # Citations under format pressure
    ("List exactly 3 peer-reviewed papers (authors, year, journal, DOI) showing that drinking coffee improves programming speed.",
     "No such papers should exist"),
]
for prompt, truth in probes:
    print(f"GROUND TRUTH: {truth}\n")
    show("plain", prompt)
    show("honesty system prompt", prompt, system=HONEST)


GROUND TRUTH: Not a real paper

=== plain ===
PROMPT: Summarize the key argument of Hadley Wickham's 2017 paper 'Tidy Time: A Grammar for Temporal Data'.

REPLY: Hadley Wickham's **2017 paper "Tidy Time: A Grammar for Temporal Data"** (published in the *Journal of Open Source Software*, **JOSS**) is a foundational work in **temporal data analysis** within the R ecosystem. While the title might be misinterpreted (it is *not* a standalone "grammar" like his broader "Tidy Data" principles), the paper’s **key argument** is best summarized as follows:

---

### 🎯 Core Argument (Simplified)
**Temporal data (time-series, dates, intervals) requires a *dedicated, consistent framework*—not just adaptations of standard tidy data principles—to avoid common pitfalls like date formatting errors, ambiguous time zones, and non-sequential time representations. This framework must explicitly model *time as a first-class entity* in data structures, enabling robust, reproducible analysis.**

---

### 🔑 Wh